# Student support: early-warning classification

**Question:** Before final results are known, which students may benefit from additional academic support?

Source: [UCI Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance). This must be used to offer support—not to punish, rank, or deny opportunity.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df=pd.read_csv(Path('../datasets/portfolio-expansion/student-performance/student/student-mat.csv'),sep=';')
# G1 and G2 are later-period grades, so excluding them makes this a genuine early-warning exercise.
y=(df.pop('G3') < 10).astype(int); X=df.drop(columns=['G1','G2'])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
num=X_train.select_dtypes('number').columns; cat=X_train.select_dtypes(exclude='number').columns
prep=ColumnTransformer([('num',SimpleImputer(strategy='median'),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
model=Pipeline([('prep',prep),('classifier',RandomForestClassifier(n_estimators=500,class_weight='balanced',min_samples_leaf=3,random_state=42))]).fit(X_train,y_train)
probability=model.predict_proba(X_test)[:,1]; prediction=(probability>=.5).astype(int)
print('PR-AUC:',round(average_precision_score(y_test,probability),3)); print(classification_report(y_test,prediction,target_names=['on track','support candidate']))

PR-AUC: 0.558
                   precision    recall  f1-score   support

         on track       0.75      0.86      0.80        66
support candidate       0.61      0.42      0.50        33

         accuracy                           0.72        99
        macro avg       0.68      0.64      0.65        99
     weighted avg       0.70      0.72      0.70        99



In [2]:
# Audit the number of students flagged at possible thresholds before choosing an intervention capacity.
thresholds=[.3,.5,.7]
display(pd.DataFrame({'threshold':thresholds,'students_flagged':[int((probability>=t).sum()) for t in thresholds],'precision':[round(y_test[probability>=t].mean(),2) if (probability>=t).any() else None for t in thresholds]}))
# Human review and a clear support pathway are required before any intervention.

,threshold,students_flagged,precision
0,0.3,71,0.39
1,0.5,23,0.61
2,0.7,0,NaN


## Portfolio takeaway

This demonstrates target-timing discipline and ethical framing. Next: validate across schools, use fairness metrics, and evaluate whether support actions improve outcomes.